# Milestone 6E: Wide-Column NoSQL Data Modeling (Delta Lake & Cassandra Emulation)

This notebook explores the **wide-column data model** tailored for high-volume AML financial analytics.

### Key Concepts:
- **Cassandra CQL Schema Design**: Partition keys for uniform distribution and clustering keys for fast time-series slices.
- **Delta Lake Physical Storage**: Emulating wide-column semantics on Databricks Community Edition.
- **Schema Evolution**: Adding new feature columns dynamically without full table rewrites.
- **ACID Upserts**: High-throughput profile updates using `MERGE INTO`.
- **Time Travel**: Historical audit logs (`VERSION AS OF`) for regulatory compliance.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.wide_column_model import create_spark_session, WideColumnStore

spark = create_spark_session('Databricks-M6E-WideColumn')
store = WideColumnStore(spark, base_storage_dir='../outputs/wide_column_store')
print('Initialized Wide-Column storage engine format:', store.format)


## 1. Inspect Cassandra Production CQL Design
Read the production Cassandra CQL schema specification ([`docs/cassandra_schema.cql`](../docs/cassandra_schema.cql)).


In [ ]:
with open('../docs/cassandra_schema.cql', 'r') as f:
    cql_text = f.read()
print('Total CQL lines:', len(cql_text.splitlines()))
print('\n'.join(cql_text.splitlines()[:35]))


## 2. Ingest Sample Data into Wide-Column Delta Tables
Create physical tables for `transaction_events` (partitioned by date) and `account_profiles` (wide row).


In [ ]:
tx_path = '../data/processed/processed_df.csv'
feat_path = '../data/processed/account_features.csv'

tx_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(tx_path).limit(10000)
feat_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(feat_path).limit(5000)

paths = store.create_tables(tx_df, feat_df)
print('Physical storage paths created:', paths)


## 3. Schema Evolution (Adding Dynamic Feature Columns)
Dynamically attach new ML risk scores to the wide-row profiles.


In [ ]:
import pyspark.sql.functions as F
new_scores = feat_df.select('account').withColumn('spark_gbt_risk_score', F.rand() * 0.9)
evolved = store.demonstrate_schema_evolution(new_scores)
evolved.select('account', 'in_degree', 'total_in', 'spark_gbt_risk_score').show(5)


## 4. Cassandra Query Emulation
Simulate single-row lookup, ego-subgraph partition scans, and materialized view queries.


In [ ]:
sample_account = feat_df.first()['account']
queries = store.simulate_cassandra_queries(sample_account)
print(f'Account Profile for {sample_account}:')
queries['single_account_lookup'].show(1)
spark.stop()
